# Release16 Guarded Reward / Checkpoint Objective

This notebook is the executable handoff from the release16 official replay and bull-reentry replay diagnostics into a guarded reward/checkpoint objective screen.

- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`
- Direct comparison baseline: `release16_release_gate_bc_official_replay_v1`
- Role: `diagnose` and prepare the next short training screen; no actor promotion is allowed here.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`
- Primary gate: relative return >= 30pp and MaxDD <= 20% on every fold
- Strict risk context: MaxDD <= 15% is retained as sidecar context, but not used as the primary pass gate in this notebook
- Changed layer: `reward`, `checkpoint_objective`, `model_selection`
- External reference: converted SOXLBacktest qlib baseline plus release16 official replay handoff

Historical success case: release16 drawdown/upside checkpoint replay got a strong `wf_2022` fold but failed full walk-forward retention.

Historical failure case: release-gate weighted BC and bull-reentry replay improved some folds but still failed at least one window, blocking actor retention.

Why this is not a duplicate: this notebook combines the latest official candidate pool with the latest bull-reentry daily paths, validates reward patch keys against `RewardConfig`, and screens a trainable guarded reward recipe before any PPO continuation.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

RUN_CWD = Path.cwd()
for candidate_root in (RUN_CWD, *RUN_CWD.parents):
    if (candidate_root / "src" / "soxl_quant").exists():
        REPO_ROOT = candidate_root
        break
else:
    raise RuntimeError(f"could not locate repo root from {RUN_CWD}")
for module_path in (REPO_ROOT / "src", REPO_ROOT):
    module_path_text = str(module_path)
    if module_path_text not in sys.path:
        sys.path.insert(0, module_path_text)
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from soxl_quant.app.drawdown_drift_reward_screen import (
    DrawdownDriftRewardScreenConfig,
    render_drawdown_drift_reward_screen_review,
    screen_drawdown_drift_reward_recipes,
    summarize_drawdown_drift_reward_screen,
)
from soxl_quant.app.reward_checkpoint_screen import (
    RewardCheckpointScreenConfig,
    render_reward_checkpoint_screen_review,
    score_reward_checkpoint_candidates,
    summarize_reward_checkpoint_screen,
)
from soxl_quant.config import RewardConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


In [2]:
EXPERIMENT = "release16_guarded_reward_checkpoint_objective_v1"
CARD_DIR = REPO_ROOT / "runtime" / "reward" / "card" / EXPERIMENT
CARD_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_DIR = REPO_ROOT / "runtime" / "model_selection" / "card" / "release16_official_replay_candidate_handoff_v1"
BULL_REENTRY_DIR = REPO_ROOT / "runtime" / "model_selection" / "card" / "release16_bull_reentry_participation_objective_replay_v1"
BULL_PREFIX = "release16_bull_reentry_participation_objective_replay_v1"

TARGET_RELATIVE_RETURN_PP = 30.0
PRIMARY_TARGET_MAX_DD_PCT = 20.0
STRICT_CONTEXT_MAX_DD_PCT = 15.0


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + chr(10), encoding="utf-8")

input_paths = {
    "official_metrics": OFFICIAL_DIR / "candidate_window_metrics.csv",
    "official_attribution": OFFICIAL_DIR / "gate_gap_attribution.csv",
    "official_summary": OFFICIAL_DIR / "summary.json",
    "bull_metrics": BULL_REENTRY_DIR / "bc_official_replay_metrics.csv",
    "bull_summary": BULL_REENTRY_DIR / "summary.json",
    "bull_selection": BULL_REENTRY_DIR / "selection.json",
}
missing_inputs = [str(path) for path in input_paths.values() if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"missing inputs: {missing_inputs}")
input_paths


{'official_metrics': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_official_replay_candidate_handoff_v1/candidate_window_metrics.csv'),
 'official_attribution': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_official_replay_candidate_handoff_v1/gate_gap_attribution.csv'),
 'official_summary': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_official_replay_candidate_handoff_v1/summary.json'),
 'bull_metrics': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_bull_reentry_participation_objective_replay_v1/bc_official_replay_metrics.csv'),
 'bull_summary': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_bull_reentry_particip

In [3]:
official_metrics = pd.read_csv(input_paths["official_metrics"])
official_attribution = pd.read_csv(input_paths["official_attribution"])
official_summary = read_json(input_paths["official_summary"])
bull_metrics = pd.read_csv(input_paths["bull_metrics"])
bull_summary = read_json(input_paths["bull_summary"])
bull_selection = read_json(input_paths["bull_selection"])

selected_bull_candidate = str(
    bull_selection.get("selected_candidate")
    or bull_summary.get("selected_candidate")
    or "bull65_hold3"
)
reference_candidate = f"{BULL_PREFIX}::release64_current_target"
selected_bull_profile = f"{BULL_PREFIX}::{selected_bull_candidate}"

print("official_metrics", official_metrics.shape)
print("official_attribution", official_attribution.shape)
print("bull_metrics", bull_metrics.shape)
print("selected_bull_candidate", selected_bull_candidate)


official_metrics (48, 18)
official_attribution (18, 10)
bull_metrics (12, 24)
selected_bull_candidate bull65_hold3


In [4]:
def classify_failure(relative_gap: float, max_dd_gap: float, gate_pass: bool) -> str:
    if bool(gate_pass):
        return "pass"
    rel = float(relative_gap)
    dd = float(max_dd_gap)
    if rel > 0 and dd > 0:
        return "joint_tradeoff"
    if dd > 0:
        return "near_miss_drawdown" if dd <= 2.0 else "drawdown_control"
    if rel > 0:
        return "underparticipation"
    return "unknown"

attr_lookup = {
    (str(row.candidate_id), str(row.window)): str(row.failure_type)
    for row in official_attribution.itertuples(index=False)
}

official_rows: list[dict[str, object]] = []
for row in official_metrics.itertuples(index=False):
    profile = str(row.candidate_id)
    window = str(row.window)
    rel_gap = float(row.relative_gap_20dd)
    dd_gap = float(row.max_dd_gap_20dd)
    gate_pass = bool(row.primary_window_gate)
    failure_type = attr_lookup.get((profile, window), classify_failure(rel_gap, dd_gap, gate_pass))
    official_rows.append(
        {
            "profile": profile,
            "source_label": str(row.source_label),
            "candidate": str(row.candidate),
            "window": window,
            "relative_return": float(row.relative_return),
            "max_dd": float(row.max_dd),
            "relative_gap": rel_gap,
            "max_dd_gap": dd_gap,
            "strict_relative_gap": float(row.relative_gap_15dd),
            "strict_max_dd_gap": float(row.max_dd_gap_15dd),
            "gate_pass": gate_pass,
            "strict_gate_pass": bool(row.strict_window_gate),
            "failure_type": failure_type,
            "evidence_source": "official_candidate_handoff",
            "daily_path": "",
        }
    )
official_gaps = pd.DataFrame(official_rows)

bull_rows: list[dict[str, object]] = []
for row in bull_metrics.reset_index(drop=True).itertuples(index=False):
    rel_gap = float(row.relative_gap_pp)
    dd_gap = float(row.maxdd_gap_20pp)
    gate_pass = bool(row.wf_gate_20dd)
    bull_rows.append(
        {
            "profile": f"{BULL_PREFIX}::{row.candidate}",
            "source_label": BULL_PREFIX,
            "candidate": str(row.candidate),
            "window": str(row.fold),
            "relative_return": float(row.relative_return_pp),
            "max_dd": float(row.max_dd_pct),
            "relative_gap": rel_gap,
            "max_dd_gap": dd_gap,
            "strict_relative_gap": float(row.relative_gap_pp),
            "strict_max_dd_gap": float(row.maxdd_gap_15pp),
            "gate_pass": gate_pass,
            "strict_gate_pass": bool(row.strict_gate_15dd),
            "failure_type": classify_failure(rel_gap, dd_gap, gate_pass),
            "evidence_source": "bull_reentry_official_daily_replay",
            "daily_path": str(row.daily_path),
        }
    )
bull_gaps = pd.DataFrame(bull_rows)

gate_gaps = pd.concat([official_gaps, bull_gaps], ignore_index=True)
required_gate_cols = {"profile", "window", "relative_gap", "max_dd_gap", "relative_return", "max_dd", "gate_pass"}
missing_gate_cols = sorted(required_gate_cols - set(gate_gaps.columns))
if missing_gate_cols:
    raise ValueError(f"gate_gaps missing columns: {missing_gate_cols}")
if gate_gaps.empty:
    raise ValueError("gate_gaps is empty")

gate_gaps_path = CARD_DIR / "gate_gaps.csv"
gate_gaps.to_csv(gate_gaps_path, index=False)

gate_gaps.groupby(["evidence_source", "failure_type"]).size().reset_index(name="rows")


,evidence_source,failure_type,rows
0,bull_reentry_official_daily_replay,drawdown_control,2
1,bull_reentry_official_daily_replay,joint_tradeoff,4
2,bull_reentry_official_daily_replay,pass,2
3,bull_reentry_official_daily_replay,underparticipation,4
4,official_candidate_handoff,drawdown_control,4
5,official_candidate_handoff,joint_tradeoff,9
6,official_candidate_handoff,near_miss_drawdown,1
7,official_candidate_handoff,pass,5
8,official_candidate_handoff,passed,6
9,official_candidate_handoff,underparticipation,23


In [5]:
handoff = official_summary.get("training_objective_handoff") or bull_summary.get("handoff") or {}
feature_groups = pd.DataFrame(
    {
        "feature_group": sorted(
            set(handoff.get("state_feature_groups", []))
            | {"trend_momentum", "relative_strength_spread", "trend_quality"}
        )
    }
)
cluster_rows: list[dict[str, object]] = []
primary_cluster = handoff.get("primary_underparticipation_cluster") or {}
if primary_cluster:
    cluster_rows.append(
        {
            "cluster_id": str(primary_cluster.get("cluster_id", "official_primary")),
            "candidate_id": str(primary_cluster.get("candidate_id", "")),
            "dominant_failure_type": str(primary_cluster.get("dominant_failure_type", "underparticipation")),
            "mean_relative_gap": float(primary_cluster.get("mean_relative_gap", 0.0) or 0.0),
            "max_dd_gap": float(primary_cluster.get("max_dd_gap", 0.0) or 0.0),
            "source": "official_training_objective_handoff",
        }
    )
selected_bull_failures = gate_gaps[
    gate_gaps["profile"].eq(selected_bull_profile) & ~gate_gaps["gate_pass"].astype(bool)
]
for row in selected_bull_failures.itertuples(index=False):
    cluster_rows.append(
        {
            "cluster_id": str(row.window),
            "candidate_id": selected_bull_profile,
            "dominant_failure_type": str(row.failure_type),
            "mean_relative_gap": float(row.relative_gap),
            "max_dd_gap": float(row.max_dd_gap),
            "source": "bull_reentry_selected_candidate",
        }
    )
poor_window_clusters = pd.DataFrame(cluster_rows)

feature_groups.to_csv(CARD_DIR / "feature_groups.csv", index=False)
poor_window_clusters.to_csv(CARD_DIR / "poor_window_clusters.csv", index=False)

print("feature groups", feature_groups["feature_group"].tolist())
display(poor_window_clusters)


feature groups ['relative_strength_spread', 'trend_momentum', 'trend_quality']


,cluster_id,candidate_id,dominant_failure_type,mean_relative_gap,max_dd_gap,source
0,wf_2020,release16_drawdown_upside_checkpoint_replay_v1...,underparticipation,30.427053,0.000000,official_training_objective_handoff
1,wf_2020,release16_bull_reentry_participation_objective...,joint_tradeoff,7.718361,1.106156,bull_reentry_selected_candidate
2,wf_2023_2024,release16_bull_reentry_participation_objective...,underparticipation,15.697938,0.000000,bull_reentry_selected_candidate


In [6]:
checkpoint_scores = score_reward_checkpoint_candidates(
    gate_gaps,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_groups,
    reference_candidate=reference_candidate,
    config=RewardCheckpointScreenConfig(
        target_relative_return=TARGET_RELATIVE_RETURN_PP,
        target_max_dd=PRIMARY_TARGET_MAX_DD_PCT,
        relative_gap_weight=1.0,
        max_dd_gap_weight=10.0,
        max_dd_drift_weight=5.0,
        relative_drift_weight=0.25,
    ),
)
checkpoint_summary = summarize_reward_checkpoint_screen(checkpoint_scores)
checkpoint_review = render_reward_checkpoint_screen_review(checkpoint_scores, checkpoint_summary)

checkpoint_scores.to_csv(CARD_DIR / "reward_checkpoint_scores.csv", index=False)
write_json(CARD_DIR / "reward_checkpoint_summary.json", checkpoint_summary)
(CARD_DIR / "reward_checkpoint_review.md").write_text(checkpoint_review, encoding="utf-8")

checkpoint_summary


{'decision': 'reward_checkpoint_screen_no_promotion',
 'selected_candidate': 'release16_bull_reentry_participation_objective_replay_v1::bull65_hold3',
 'selected_reward_checkpoint_loss': 34.47785927885962,
 'selected_gate_gap_loss': 34.47785927885962,
 'selected_failure_mix': 'joint_tradeoff:1,pass:1,underparticipation:1',
 'selected_min_relative_return': 14.302062458265096,
 'selected_max_max_dd': 21.10615606724233,
 'promotion_allowed': False,
 'ppo_allowed_next': False,
 'required_objective_terms': ['checkpoint_objective_loss',
  'drawdown_drift_penalty',
  'trend_preservation_floor'],
 'next_mechanism': 'train_drawdown_drift_reward_or_checkpoint_recipe'}

In [7]:
selected_bull_metrics = bull_metrics[bull_metrics["candidate"].astype(str).eq(selected_bull_candidate)].copy().reset_index(drop=True)
if selected_bull_metrics.empty:
    raise ValueError(f"selected bull candidate has no metrics: {selected_bull_candidate}")

daily_paths = {}
for row in selected_bull_metrics.itertuples(index=False):
    path = REPO_ROOT / str(row.daily_path)
    if not path.exists():
        raise FileNotFoundError(path)
    # Existing helper parses the window from the final segment after `__`.
    daily_paths[f"{selected_bull_candidate}__{row.fold}"] = pd.read_csv(path)

recipe_scores = screen_drawdown_drift_reward_recipes(
    daily_paths,
    gate_gaps=gate_gaps[gate_gaps["profile"].eq(selected_bull_profile)].copy(),
    selected_candidate=selected_bull_profile,
    config=DrawdownDriftRewardScreenConfig(
        target_relative_return_pp=TARGET_RELATIVE_RETURN_PP,
        target_max_dd_pct=PRIMARY_TARGET_MAX_DD_PCT,
        min_drawdown_failure_capture=0.50,
        min_trend_underparticipation_capture=0.50,
        max_overreach_share=0.25,
    ),
)
recipe_summary = summarize_drawdown_drift_reward_screen(recipe_scores)
recipe_review = render_drawdown_drift_reward_screen_review(recipe_scores, recipe_summary, top_n=10)

recipe_scores.to_csv(CARD_DIR / "reward_recipe_scores.csv", index=False)
write_json(CARD_DIR / "reward_recipe_summary.json", recipe_summary)
(CARD_DIR / "reward_recipe_review.md").write_text(recipe_review, encoding="utf-8")
reward_patch = recipe_summary.get("reward_objective_patch", {})
write_json(CARD_DIR / "reward_objective_patch.json", reward_patch)

reward_patch


{'drawdown_drift_penalty': 1.5,
 'drawdown_drift_position_threshold': 0.15,
 'drawdown_drift_reward_enabled': True,
 'drawdown_drift_threshold_pct': 18.0,
 'trend_preservation_floor_enabled': True,
 'trend_preservation_floor_penalty': 1.0,
 'trend_preservation_floor_position': 0.25,
 'trend_preservation_lookback': 21,
 'trend_preservation_min_buyhold_return_pct': 5.0}

In [8]:
reward_config_fields = set(RewardConfig.__dataclass_fields__)
missing_patch_keys = sorted(set(reward_patch) - reward_config_fields)
if missing_patch_keys:
    raise ValueError(f"reward patch keys not in RewardConfig: {missing_patch_keys}")

required_terms = set(checkpoint_summary.get("required_objective_terms", []))
term_readiness = {
    "checkpoint_objective_loss": True,
    "drawdown_drift_penalty": bool(reward_patch.get("drawdown_drift_reward_enabled"))
    and float(reward_patch.get("drawdown_drift_penalty", 0.0)) > 0.0,
    "trend_preservation_floor": bool(reward_patch.get("trend_preservation_floor_enabled"))
    and float(reward_patch.get("trend_preservation_floor_penalty", 0.0)) > 0.0,
    "short_volshock_state_deleverage_consistency": False,
}
missing_ready_terms = sorted(term for term in required_terms if not term_readiness.get(term, False))

selected_checkpoint_row = checkpoint_scores.iloc[0].to_dict() if not checkpoint_scores.empty else {}
selected_recipe_row = recipe_scores.iloc[0].to_dict() if not recipe_scores.empty else {}
selected_bull_gate = gate_gaps[gate_gaps["profile"].eq(selected_bull_profile)].copy()

strict_gate_count = int(selected_bull_gate["strict_gate_pass"].astype(bool).sum()) if not selected_bull_gate.empty else 0
primary_gate_count = int(selected_bull_gate["gate_pass"].astype(bool).sum()) if not selected_bull_gate.empty else 0
window_count = int(len(selected_bull_gate))

selection = {
    "experiment": EXPERIMENT,
    "selected_bull_candidate": selected_bull_candidate,
    "selected_bull_profile": selected_bull_profile,
    "selected_checkpoint_candidate": str(checkpoint_summary.get("selected_candidate", "")),
    "selected_recipe": str(recipe_summary.get("selected_recipe", "")),
    "reward_patch_valid_for_reward_config": not missing_patch_keys,
    "missing_patch_keys": missing_patch_keys,
    "missing_ready_required_terms": missing_ready_terms,
    "primary_gate_count_20dd": primary_gate_count,
    "strict_gate_count_15dd": strict_gate_count,
    "window_count": window_count,
    "short_training_allowed_next": bool(recipe_summary.get("training_allowed_next", False)) and not missing_ready_terms,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD_DIR / "selection.json", selection)
selection


{'experiment': 'release16_guarded_reward_checkpoint_objective_v1',
 'selected_bull_candidate': 'bull65_hold3',
 'selected_bull_profile': 'release16_bull_reentry_participation_objective_replay_v1::bull65_hold3',
 'selected_checkpoint_candidate': 'release16_bull_reentry_participation_objective_replay_v1::bull65_hold3',
 'selected_recipe': 'dd1.5_thr18_pos0.15_tf1_floor0.25_lb21',
 'reward_patch_valid_for_reward_config': True,
 'missing_patch_keys': [],
 'missing_ready_required_terms': [],
 'primary_gate_count_20dd': 1,
 'strict_gate_count_15dd': 0,
 'window_count': 3,
 'short_training_allowed_next': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False}

In [9]:
def _safe_float(value: object, default: float = 0.0) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return default
    return out if np.isfinite(out) else default

candidate_summary = (
    gate_gaps.groupby("profile", as_index=False)
    .agg(
        window_count=("window", "count"),
        primary_gate_count_20dd=("gate_pass", "sum"),
        strict_gate_count_15dd=("strict_gate_pass", "sum"),
        min_relative_return_pp=("relative_return", "min"),
        max_max_dd_pct=("max_dd", "max"),
        total_relative_gap_20dd=("relative_gap", "sum"),
        total_max_dd_gap_20dd=("max_dd_gap", "sum"),
    )
    .sort_values(["primary_gate_count_20dd", "total_relative_gap_20dd", "total_max_dd_gap_20dd"], ascending=[False, True, True])
)
candidate_summary.to_csv(CARD_DIR / "candidate_gate_summary.csv", index=False)

def rel(path: Path) -> str:
    return str(path.relative_to(REPO_ROOT))

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": (
        "guarded_reward_recipe_ready_for_short_training_no_promotion"
        if selection["short_training_allowed_next"]
        else "guarded_reward_recipe_not_ready_no_promotion"
    ),
    "checkpoint_decision": checkpoint_summary.get("decision", ""),
    "recipe_decision": recipe_summary.get("decision", ""),
    "selected_bull_candidate": selected_bull_candidate,
    "selected_checkpoint_candidate": checkpoint_summary.get("selected_candidate", ""),
    "selected_recipe": recipe_summary.get("selected_recipe", ""),
    "selected_reward_checkpoint_loss": _safe_float(checkpoint_summary.get("selected_reward_checkpoint_loss", 0.0)),
    "selected_reward_recipe_loss": _safe_float(recipe_summary.get("selected_reward_recipe_loss", 0.0)),
    "drawdown_failure_capture": _safe_float(recipe_summary.get("drawdown_failure_capture", 0.0)),
    "trend_underparticipation_capture": _safe_float(recipe_summary.get("trend_underparticipation_capture", 0.0)),
    "overreach_share": _safe_float(recipe_summary.get("overreach_share", 0.0)),
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": PRIMARY_TARGET_MAX_DD_PCT,
    "strict_context_max_dd_pct": STRICT_CONTEXT_MAX_DD_PCT,
    "selected_primary_gate_count_20dd": primary_gate_count,
    "selected_strict_gate_count_15dd": strict_gate_count,
    "selected_window_count": window_count,
    "required_objective_terms": checkpoint_summary.get("required_objective_terms", []),
    "missing_ready_required_terms": missing_ready_terms,
    "reward_objective_patch": reward_patch,
    "short_training_allowed_next": selection["short_training_allowed_next"],
    "ppo_allowed_next": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_mechanism": (
        "short_foldlocal_bc_or_ppo_training_screen_with_guarded_reward_patch"
        if selection["short_training_allowed_next"]
        else "revise_state_features_or_checkpoint_objective_before_training"
    ),
    "source_context": {
        "official_handoff_decision": official_summary.get("decision", ""),
        "bull_reentry_decision": bull_summary.get("decision", ""),
        "canonical_baseline": official_summary.get("canonical_baseline", bull_summary.get("canonical_baseline", "")),
        "direct_comparison_baseline": official_summary.get("direct_comparison_baseline", bull_summary.get("direct_comparison_baseline", "")),
        "soxlbacktest_reference": bull_summary.get("output_paths", {}).get("soxlbacktest_reference_metrics", ""),
    },
    "output_paths": {
        "notebook": rel(CARD_DIR / f"{EXPERIMENT}.ipynb"),
        "gate_gaps": rel(CARD_DIR / "gate_gaps.csv"),
        "candidate_gate_summary": rel(CARD_DIR / "candidate_gate_summary.csv"),
        "reward_checkpoint_scores": rel(CARD_DIR / "reward_checkpoint_scores.csv"),
        "reward_checkpoint_summary": rel(CARD_DIR / "reward_checkpoint_summary.json"),
        "reward_recipe_scores": rel(CARD_DIR / "reward_recipe_scores.csv"),
        "reward_recipe_summary": rel(CARD_DIR / "reward_recipe_summary.json"),
        "reward_objective_patch": rel(CARD_DIR / "reward_objective_patch.json"),
        "selection": rel(CARD_DIR / "selection.json"),
        "summary": rel(CARD_DIR / "summary.json"),
        "conclusion": rel(CARD_DIR / "conclusion.md"),
    },
    "conclusion": [],
}
summary["conclusion"] = [
    f"Selected bull-reentry candidate `{selected_bull_candidate}` still has {primary_gate_count}/{window_count} primary 20% walk-forward gate passes and {strict_gate_count}/{window_count} strict 15% passes; no actor is promoted or retained.",
    f"Checkpoint screen selected `{summary['selected_checkpoint_candidate']}` with decision `{summary['checkpoint_decision']}`; this remains a pre-training diagnostic.",
    f"Reward recipe `{summary['selected_recipe']}` is {'ready' if selection['short_training_allowed_next'] else 'not ready'} for a short fold-local training screen, with drawdown capture {summary['drawdown_failure_capture']:.1%}, trend-underparticipation capture {summary['trend_underparticipation_capture']:.1%}, and overreach {summary['overreach_share']:.1%}.",
    "The emitted reward patch validates against RewardConfig and is not a runtime overlay or retained model.",
]
write_json(CARD_DIR / "summary.json", summary)
summary


{'experiment': 'release16_guarded_reward_checkpoint_objective_v1',
 'created_at': '2026-06-14T20:53:17.075663+00:00',
 'decision': 'guarded_reward_recipe_ready_for_short_training_no_promotion',
 'checkpoint_decision': 'reward_checkpoint_screen_no_promotion',
 'recipe_decision': 'drawdown_drift_reward_recipe_ready_for_short_training_screen',
 'selected_bull_candidate': 'bull65_hold3',
 'selected_checkpoint_candidate': 'release16_bull_reentry_participation_objective_replay_v1::bull65_hold3',
 'selected_recipe': 'dd1.5_thr18_pos0.15_tf1_floor0.25_lb21',
 'selected_reward_checkpoint_loss': 34.47785927885962,
 'selected_reward_recipe_loss': 1.5071085999435105,
 'drawdown_failure_capture': 1.0,
 'trend_underparticipation_capture': 1.0,
 'overreach_share': 0.0,
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'strict_context_max_dd_pct': 15.0,
 'selected_primary_gate_count_20dd': 1,
 'selected_strict_gate_count_15dd': 0,
 'selected_window_count': 3,
 'required_objective_terms'

In [10]:
conclusion_md = chr(10).join(
    [
        "# Conclusion",
        "",
        f"- decision: `{summary['decision']}`",
        f"- selected_bull_candidate: `{selected_bull_candidate}`",
        f"- selected_recipe: `{summary['selected_recipe']}`",
        f"- short_training_allowed_next: `{summary['short_training_allowed_next']}`",
        f"- promotion_allowed: `{summary['promotion_allowed']}`",
        f"- actor_retention_allowed: `{summary['actor_retention_allowed']}`",
        "",
        "## Readout",
        "",
        *[f"- {line}" for line in summary["conclusion"]],
        "",
        "## Next Action",
        "",
        f"- `{summary['next_mechanism']}`",
    ]
) + chr(10)
(CARD_DIR / "conclusion.md").write_text(conclusion_md, encoding="utf-8")
display(Markdown(conclusion_md))


# Conclusion

- decision: `guarded_reward_recipe_ready_for_short_training_no_promotion`
- selected_bull_candidate: `bull65_hold3`
- selected_recipe: `dd1.5_thr18_pos0.15_tf1_floor0.25_lb21`
- short_training_allowed_next: `True`
- promotion_allowed: `False`
- actor_retention_allowed: `False`

## Readout

- Selected bull-reentry candidate `bull65_hold3` still has 1/3 primary 20% walk-forward gate passes and 0/3 strict 15% passes; no actor is promoted or retained.
- Checkpoint screen selected `release16_bull_reentry_participation_objective_replay_v1::bull65_hold3` with decision `reward_checkpoint_screen_no_promotion`; this remains a pre-training diagnostic.
- Reward recipe `dd1.5_thr18_pos0.15_tf1_floor0.25_lb21` is ready for a short fold-local training screen, with drawdown capture 100.0%, trend-underparticipation capture 100.0%, and overreach 0.0%.
- The emitted reward patch validates against RewardConfig and is not a runtime overlay or retained model.

## Next Action

- `short_foldlocal_bc_or_ppo_training_screen_with_guarded_reward_patch`
